## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161394?b2cUser=true"><b>Langchain Agentes - Criação do Agente com a Ferramenta</b></a><br/>

<b>Objetivo:</b> Criação do agente que chamará a ferramenta.<br/>
<ul><li>Maneira pela qual a LLM toma ciência da ferramenta.</li></ul>

<b>PASSOS:</b><br/>
<ul>
    <b><li>CRIAÇÃO DAS FERRAMENTAS</li></b><br/>
    <ul>
        <ol>
            <li>Criação da Ferramenta DadosDeEstudante (Refinando a anterior)</li>
            <li>Instanciando a Ferramenta que a LLM precisa usar</li>
        </ol>
    </ul><br/>
    <b><li>CRIAÇÃO DO AGENTE</li></b><br/>   
    <ul>
        <ol>
            <li>Informando para a LLM as ferramentas que eu tenho (Usa a ferramenta que foi instanciada)</li>
            <li>Executando o Agente</li>
        </ol>
    </ul><br/>
</ul>

In [10]:
#%pip install -r requirements.txt

In [11]:
from os import getenv
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
#from langchain.globals import set_debug

#set_debug(True)

load_dotenv()

llm = ChatOpenAI(
                    model="gpt-5-mini",
                    api_key=getenv("API_KEY")            
                )

class ExtratordeEstudante(BaseModel):
    estudante: str = Field(description="Nome do estudante informado, sempre em letras minúsculas. Exemplo: joão, carla, joana")

### <b>CRIAÇÃO DE FERRAMENTAS</b>
Que ferramentas eu tenho disponíveis para se obter os dados da Ana ?

<b>1) Criação da Ferramenta DadosDeEstudante</b>
<ul><li> A classe deve estender de BaseTool</li></ul>

In [12]:
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


class DadosDeEstudante(BaseTool):
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                        """ # Descrição da ferramenta 
    
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:        
        
        parseador = JsonOutputParser(pydantic_object=ExtratordeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    Você deve analisar a {input} e extrair o nome de estudante informado.
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["input"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | llm | parseador
        
        resposta = cadeia.invoke({"input": input})
        
        return resposta['estudante']


<b>2) Instanciando a Ferramenta que a LLM precisa usar</b>

In [13]:
from langchain.agents import Tool

dados_de_estudante = DadosDeEstudante() # OBJETO DA MINHA FERRAMENTA

# MATRIZ DE FERRAMENTAS
tools = [
            # Instanciando ferramentas
            Tool(
                    name=dados_de_estudante.name,
                    func=dados_de_estudante.run,
                    description=dados_de_estudante.description                
            )
]

### <b>CRIAÇÃO DO AGENTE</b>

<b>3) Informando para a LLM as ferramentas que eu tenho</b> 
<ul><li>Para isso, é necessário criar um agente com as ferramentas</li></ul>

In [14]:
from langchain.agents import create_openai_tools_agent
from langchain import hub
import warnings

warnings.filterwarnings("ignore")


# CRIANDO UM AGENTE COM AS FERRAMENTAS
prompt=(hub.pull(owner_repo_commit="hwchase17/openai-functions-agent"))

agente = create_openai_tools_agent(
                                    llm=llm, # INFORMA A LLM QUE VAI SER USADA PELO AGENTE
                                    tools=tools, # PASSANDO PARA A LLM A FERRAMENTA QUE ELA PODE USAR. INSTÂNCIA DA FERRAMENTA
                                    prompt=prompt # PROMPT DE INICIALIZAÇÃO PARA INFORMAR PARA A LLM SOBRE A FERRAMENTA. 
                                                  # JÁ EXISTEM PROMPTS PRONTOS NO REPOSITÓRIO DO LANGSMITH, DE ACORDO COM O TIPO DE FERRAMENTA. 
                                                        # Para agente de função (https://smith.langchain.com/hub/hwchase17/openai-functions-agent)
                                                                 
                                  )

print(prompt)


input_variables=['agent_scratchpad', 'input'] optional_variables=['chat_history'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

<b>4) Executando o agente</b>

In [15]:
from langchain.agents import AgentExecutor

executor = AgentExecutor(
                            agent=agente, 
                            tools=tools,
                            verbose=True
                        )

pergunta = "Quais são os dados da Ana?"

resposta = executor.invoke({"input": pergunta})

print(resposta)



> Entering new AgentExecutor chain...

Invoking: `dados_de_estudante` with `Ana`


anaPreciso de um pouco mais de informação.

A busca automática retornou apenas o nome "ana" — sem detalhes. Você pode me dizer:
- Qual Ana? (sobrenome ou número de matrícula)
- Quais dados você quer ver? (ex.: contato, notas, faltas, turma/curso, histórico escolar, responsáveis)
- Você tem autorização para acessar esses dados? (por questões de privacidade, só posso mostrar informações completas se você for a própria Ana, um responsável legal ou tiver permissão)

Diga como prefere que eu continue (fornecer sobrenome/matrícula ou autorizar o acesso) e eu tento buscar os dados completos.

> Finished chain.
{'input': 'Quais são os dados da Ana?', 'output': 'Preciso de um pouco mais de informação.\n\nA busca automática retornou apenas o nome "ana" — sem detalhes. Você pode me dizer:\n- Qual Ana? (sobrenome ou número de matrícula)\n- Quais dados você quer ver? (ex.: contato, notas, faltas, turma/curso, histó

A saída "ana", já comprova que a LLM usou a nossa ferramenta